# 💰 Análisis de Costes de Rotación - EBLET v2.0

## Objetivo
Cuantificar el impacto económico del burnout, boreout y bienestar.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

import os
import sys
RAIZ_PROYECTO = r"C:\Users\torre\OneDrive\Escritorio\EBLET-People-Analytics\Python"
sys.path.insert(0, os.path.join(RAIZ_PROYECTO, "src"))

from costes_rotacion import (
    calcular_costes_escenario,
    calcular_costes_empresa,
    generar_resumen_ejecutivo,
    calcular_roi_intervencion
)

print("✅ Notebook de Análisis de Costes cargado")

✅ Notebook de Análisis de Costes cargado


## 1. Resumen Ejecutivo

In [2]:
ESCENARIOS = ["saludable", "estable", "riesgo_burnout", "riesgo_boreout", "critico"]

ETIQUETAS = {
    "saludable": "🟢 Saludable",
    "estable": "🟡 Estable",
    "riesgo_burnout": "🟠 Riesgo Burnout",
    "riesgo_boreout": "🔵 Riesgo Boreout",
    "critico": "🔴 Crítico"
}

COLORES = {
    "saludable": "#2ecc71",
    "estable": "#f1c40f",
    "riesgo_burnout": "#e67e22",
    "riesgo_boreout": "#3498db",
    "critico": "#e74c3c"
}

dfs = {}
for esc in ESCENARIOS:
    ruta = os.path.join(RAIZ_PROYECTO, f"datasets/{esc}/empleados.csv")
    df = pd.read_csv(ruta)
    df["escenario"] = esc
    df["escenario_label"] = ETIQUETAS[esc]
    dfs[esc] = df

df_all = pd.concat(dfs.values(), ignore_index=True)

resumen = generar_resumen_ejecutivo(df_all)

print("="*70)
print("💰 RESUMEN EJECUTIVO - COSTES DE ROTACIÓN")
print("="*70)
print(f"\n👥 Total empleados: {resumen['total_empleados']:,}")
print(f"🏢 Total empresas: {resumen['total_empresas']}")
print(f"\n💸 COSTES:")
print(f"   Coste total: {resumen['coste_total_rotacion']:>15,.2f}€")
print(f"   Coste medio/empleado: {resumen['coste_medio_por_empleado']:>12,.2f}€")
print(f"   Masa salarial: {resumen['masa_salarial_total']:>18,.2f}€")
print(f"   % sobre masa salarial: {resumen['coste_pct_masa_salarial']:.2f}%")
print(f"\n🎯 AHORRO POTENCIAL (30% reducción): {resumen['ahorro_potencial_30']:,.2f}€")
print("="*70)

💰 RESUMEN EJECUTIVO - COSTES DE ROTACIÓN

👥 Total empleados: 12,500
🏢 Total empresas: 200

💸 COSTES:
   Coste total:  357,589,132.67€
   Coste medio/empleado:    28,607.13€
   Masa salarial:     810,197,941.00€
   % sobre masa salarial: 44.14%

🎯 AHORRO POTENCIAL (30% reducción): 107,276,739.80€


## 2. Costes por Escenario

In [7]:
# =====================================================
# CARGA DE DATOS
# =====================================================

ESCENARIOS = ["saludable", "estable", "riesgo_burnout", "riesgo_boreout", "critico"]

ETIQUETAS = {
    "saludable": "🟢 Saludable",
    "estable": "🟡 Estable",
    "riesgo_burnout": "🟠 Riesgo Burnout",
    "riesgo_boreout": "🔵 Riesgo Boreout",
    "critico": "🔴 Crítico"
}

COLORES = {
    "saludable": "#2ecc71",
    "estable": "#f1c40f",
    "riesgo_burnout": "#e67e22",
    "riesgo_boreout": "#3498db",
    "critico": "#e74c3c"
}

# Cargar todos los datasets
dfs = {}
for esc in ESCENARIOS:
    ruta = os.path.join(RAIZ_PROYECTO, f"datasets/{esc}/empleados.csv")
    df = pd.read_csv(ruta)
    df["escenario"] = esc
    df["escenario_label"] = ETIQUETAS[esc]
    dfs[esc] = df  # ← ESTA LÍNEA ES CRÍTICA

# Unificar todos los datasets
df_all = pd.concat(dfs.values(), ignore_index=True)

print(f"✅ Total empleados: {len(df_all):,}")
print(f"✅ Total empresas: {df_all['empresa_id'].nunique()}")

✅ Total empleados: 12,500
✅ Total empresas: 200


In [10]:
# =====================================================
# GRÁFICO: COSTE TOTAL POR ESCENARIO
# =====================================================

# Crear columna escenario_nombre que espera la función
df_all["escenario_nombre"] = df_all["escenario"].map({
    "saludable": "Saludable",
    "estable": "Estable",
    "riesgo_burnout": "Riesgo Burnout",
    "riesgo_boreout": "Riesgo Boreout",
    "critico": "Crítico"
})

# Calcular costes por escenario
costes_esc = calcular_costes_escenario(df_all)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=costes_esc["escenario"],
    y=costes_esc["coste_total"],
    marker_color=[COLORES[esc] for esc in ESCENARIOS],
    text=[f"{v:,.0f}€" for v in costes_esc["coste_total"]],
    textposition='outside'
))

fig.update_layout(
    title="💰 Coste Total de Rotación por Escenario",
    xaxis_title="Escenario Organizacional",
    yaxis_title="Coste Anual Estimado (€)",
    height=500,
    yaxis_tickformat=",d"
)
fig.show()

print("\n📊 Análisis:")
print(f"   - Crítico: mayor coste")
print(f"   - Saludable: menor coste")
print(f"   - Diferencia: {costes_esc['coste_total'].max() - costes_esc['coste_total'].min():,.0f}€")


📊 Análisis:
   - Crítico: mayor coste
   - Saludable: menor coste
   - Diferencia: 67,195,867€


## 3. Scatter: Burnout vs Coste

In [4]:
df_sample = df_all.sample(n=2000, random_state=42)

fig = px.scatter(
    df_sample,
    x="kpi_burnout",
    y="coste_rotacion_individual",
    color="escenario_label",
    color_discrete_map={ETIQUETAS[k]: v for k, v in COLORES.items()},
    hover_data=["seniority", "salario", "departamento"],
    opacity=0.6,
    title="🔥 Burnout vs Coste de Rotación",
    labels={
        "kpi_burnout": "KPI Burnout",
        "coste_rotacion_individual": "Coste (€)"
    }
)
fig.update_layout(height=600)
fig.show()

## 4. Coste por Seniority

In [5]:
coste_seniority = df_all.groupby("seniority").agg({
    "coste_rotacion_individual": ["mean", "sum"]
}).reset_index()
coste_seniority.columns = ["seniority", "coste_medio", "coste_total"]

fig = px.bar(
    coste_seniority,
    x="seniority",
    y="coste_total",
    title="💼 Coste Total de Rotación por Seniority",
    color="coste_total",
    color_continuous_scale="Reds",
    text=[f"{v:,.0f}€" for v in coste_seniority["coste_total"]]
)
fig.update_layout(height=500)
fig.show()

## 5. Análisis ROI de Intervención

In [6]:
print("="*70)
print("📈 ANÁLISIS ROI DE INTERVENCIÓN")
print("="*70)

df_critico = dfs["critico"]
coste_actual = df_critico["coste_rotacion_individual"].sum()

print(f"\n🔴 Escenario Crítico:")
print(f"   Coste actual: {coste_actual:,.2f}€")

intervenciones = [
    {"nombre": "Programa básico", "coste": 50000, "reduccion": 0.20},
    {"nombre": "Programa integral", "coste": 150000, "reduccion": 0.35},
    {"nombre": "Transformación completa", "coste": 300000, "reduccion": 0.50}
]

print("\n💡 Escenarios de intervención:")
for interv in intervenciones:
    roi = calcular_roi_intervencion(
        coste_actual, interv["coste"], interv["reduccion"]
    )
    print(f"\n   {interv['nombre']}:")
    print(f"      Inversión: {interv['coste']:,.0f}€")
    print(f"      Reducción: {interv['reduccion']*100:.0f}%")
    print(f"      Ahorro: {roi['ahorro_estimado']:,.0f}€")
    print(f"      ROI: {roi['roi_porcentaje']:.1f}%")
    print(f"      Payback: {roi['payback_meses']} meses")

📈 ANÁLISIS ROI DE INTERVENCIÓN

🔴 Escenario Crítico:
   Coste actual: 104,716,023.40€

💡 Escenarios de intervención:

   Programa básico:
      Inversión: 50,000€
      Reducción: 20%
      Ahorro: 20,943,205€
      ROI: 41786.4%
      Payback: 0.0 meses

   Programa integral:
      Inversión: 150,000€
      Reducción: 35%
      Ahorro: 36,650,608€
      ROI: 24333.7%
      Payback: 0.0 meses

   Transformación completa:
      Inversión: 300,000€
      Reducción: 50%
      Ahorro: 52,358,012€
      ROI: 17352.7%
      Payback: 0.1 meses


## 📝 Conclusiones

1. 💰 El coste total de rotación es significativo
2. 📊 Los escenarios críticos multiplican el coste
3. 👥 Los perfiles Senior/Lead tienen mayor coste individual
4. 🎯 Las intervenciones tienen ROI positivo demostrado
5. 🔬 La prevención es más rentable que la corrección